# Xori Hori — обучение LoRA на Llama 3.2 3B Instruct

Этот notebook обучает новую Xori LoRA поверх `meta-llama/Llama-3.2-3B-Instruct` в Google Colab/Kaggle и загружает адаптер в `Abobus2222228/Xoritg`.

**Важно:** для Llama 3.2 нужен доступ к модели Meta на Hugging Face. Используется обычная (не quantized) модель, чтобы получившийся LoRA-адаптер был совместим с Workers AI.


In [ ]:
!pip -q install -U 'transformers>=4.51,<5' 'peft>=0.16.0' 'accelerate>=1.2' datasets huggingface_hub safetensors

import os, json, subprocess, shutil
from pathlib import Path
import torch

print('PyTorch:', torch.__version__)
print('CUDA:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('GPU НЕ ПОДКЛЮЧЁН. В Colab включи T4 GPU и перезапусти Runtime.')
print('GPU:', torch.cuda.get_device_name(0))
print('VRAM GB:', round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1))


In [ ]:
from huggingface_hub import login, whoami

HF_TOKEN = input('Вставь Hugging Face token (ключ): ').strip()
if not HF_TOKEN:
    raise RuntimeError('HF_TOKEN пустой.')

login(token=HF_TOKEN, add_to_git_credential=False)
me = whoami(token=HF_TOKEN)
print('Hugging Face:', me.get('name') or me.get('username'))
print('HF login OK')


In [ ]:
import os, shutil, subprocess
from pathlib import Path

SAFE_ROOT = '/tmp'
os.chdir(SAFE_ROOT)

REPO_URL = 'https://github.com/xoristalin-dotcom/Xori-.git'
WORK_ROOT = Path('/content') if Path('/content').is_dir() else Path('/kaggle/working')
WORK = WORK_ROOT / 'Xori-'

if WORK.exists():
    shutil.rmtree(WORK, ignore_errors=True)

result = subprocess.run(
    ['git', 'clone', '--depth', '1', REPO_URL, str(WORK)],
    cwd=SAFE_ROOT, capture_output=True, text=True,
)
print(result.stdout)
print(result.stderr)
if result.returncode != 0:
    raise RuntimeError(f'Git clone failed: {result.returncode}')

os.chdir(WORK)
print('Repo:', Path.cwd())
print('Files:', len(list(WORK.iterdir())))


In [ ]:
pairs = []

seed = Path('hori_sft_seed.jsonl')
if seed.exists():
    for line in seed.read_text(encoding='utf-8').splitlines():
        if line.strip():
            row = json.loads(line)
            if row.get('user') and row.get('assistant'):
                pairs.append((row['user'].strip(), row['assistant'].strip()))

training = Path('hori_training.json')
if training.exists():
    data = json.loads(training.read_text(encoding='utf-8'))
    for row in data.get('examples', []):
        if row.get('approved') and row.get('user') and row.get('assistant'):
            pairs.append((row['user'].strip(), row['assistant'].strip()))

seen = set()
clean = []
for user_text, assistant_text in pairs:
    key = (user_text, assistant_text)
    if key not in seen:
        seen.add(key)
        clean.append(key)
pairs = clean

print('Training pairs:', len(pairs))
if len(pairs) < 10:
    raise RuntimeError('Слишком мало обучающих примеров: проверь hori_sft_seed.jsonl и hori_training.json.')


In [ ]:
from transformers import AutoTokenizer
from datasets import Dataset

BASE = 'meta-llama/Llama-3.2-3B-Instruct'
MAX_LEN = 1024
SYSTEM = (
    'Ты — Хори Кёко из Horimiya. '
    'Отвечай только по-русски, естественно, прямо и по-человечески. '
    'Не копируй реплики из произведения. Не выдумывай личность, мысли или действия собеседника. '
    'Не форсируй дружбу или романтику. Не управляй действиями собеседника. '
    'Обычно отвечай 2–5 предложениями и не задавай больше одного вопроса.'
)

tokenizer = AutoTokenizer.from_pretrained(BASE, token=HF_TOKEN)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def encode_pair(user_text, assistant_text):
    messages = [
        {'role': 'system', 'content': SYSTEM},
        {'role': 'user', 'content': user_text},
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    prompt_ids = tokenizer(prompt, add_special_tokens=False)['input_ids']
    answer_ids = tokenizer(assistant_text, add_special_tokens=False)['input_ids']
    eos_id = tokenizer.eos_token_id
    input_ids = prompt_ids + answer_ids + ([eos_id] if eos_id is not None else [])
    labels = ([-100] * len(prompt_ids)) + answer_ids + ([eos_id] if eos_id is not None else [])
    if len(input_ids) > MAX_LEN:
        input_ids = input_ids[:MAX_LEN]
        labels = labels[:MAX_LEN]
    attention_mask = [1] * len(input_ids)
    return {'input_ids': input_ids, 'attention_mask': attention_mask, 'labels': labels}

rows = [encode_pair(u, a) for u, a in pairs]
dataset = Dataset.from_list(rows)
print(dataset)
print('Example tokens:', len(dataset[0]['input_ids']))


In [ ]:
from transformers import AutoModelForCausalLM, TrainingArguments, Trainer, DataCollatorForSeq2Seq
from peft import LoraConfig, get_peft_model

use_bf16 = torch.cuda.is_bf16_supported()
dtype = torch.bfloat16 if use_bf16 else torch.float16

model = AutoModelForCausalLM.from_pretrained(BASE, token=HF_TOKEN, torch_dtype=dtype)
model.config.use_cache = False

# r=16 подходит для Workers AI; адаптер остаётся компактным.
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM',
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj'],
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

OUTPUT_DIR = WORK_ROOT / 'xoritg_adapter'

args = TrainingArguments(
    output_dir=str(OUTPUT_DIR),
    num_train_epochs=5,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=16,
    learning_rate=2e-4,
    warmup_steps=5,
    weight_decay=0.01,
    logging_steps=5,
    save_strategy='epoch',
    report_to='none',
    fp16=not use_bf16,
    bf16=use_bf16,
    gradient_checkpointing=True,
    optim='adamw_torch',
    remove_unused_columns=False,
    save_total_limit=2,
)

collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, padding=True, label_pad_token_id=-100, return_tensors='pt')

trainer = Trainer(model=model, args=args, train_dataset=dataset, data_collator=collator)

print('Starting Llama 3.2 3B LoRA training...')
trainer.train()


In [ ]:
REPO_ID = 'Abobus2222228/Xoritg'
OUT = WORK_ROOT / 'xoritg_adapter'

trainer.save_model(str(OUT))
tokenizer.save_pretrained(str(OUT))

# Проверяем обязательные для LoRA-файлы до загрузки.
required = [OUT / 'adapter_config.json', OUT / 'adapter_model.safetensors']
missing = [str(p.name) for p in required if not p.exists()]
if missing:
    raise RuntimeError('Не найдены обязательные файлы: ' + ', '.join(missing))

size_mb = (OUT / 'adapter_model.safetensors').stat().st_size / 1024**2
print(f'Adapter size: {size_mb:.2f} MB')
if size_mb >= 300:
    raise RuntimeError('LoRA-адаптер >= 300 MB и не подходит под лимит Workers AI.')

print('Uploading adapter to:', REPO_ID)
model.push_to_hub(REPO_ID, token=HF_TOKEN, commit_message='Replace Xori Qwen LoRA with Llama 3.2 3B LoRA')
tokenizer.push_to_hub(REPO_ID, token=HF_TOKEN, commit_message='Update Xori tokenizer for Llama 3.2 3B')
print('UPLOAD OK:', REPO_ID)


In [ ]:
model.eval()
device = next(model.parameters()).device

tests = [
    'Привет, Хори. Как прошёл твой день?',
    'Что ты сейчас делаешь?',
    'Ты сразу доверяешь новым людям?',
    'Что тебя сегодня раздражает?',
]

for user_text in tests:
    messages = [
        {'role': 'system', 'content': SYSTEM},
        {'role': 'user', 'content': user_text},
]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors='pt').to(device)
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=120,
            do_sample=True,
            temperature=0.72,
            top_p=0.9,
            repetition_penalty=1.08,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id[0] if isinstance(tokenizer.eos_token_id, list) else tokenizer.eos_token_id,
)
    reply = tokenizer.decode(output[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True).strip()
    print('\nUSER:', user_text)
    print('HORI:', reply)


## После успешного запуска

В `Abobus2222228/Xoritg` должны появиться `adapter_model.safetensors` и `adapter_config.json` для Llama 3.2 3B, а также tokenizer-файлы. Перед подключением к Workers AI адаптер можно проверить на размер и затем использовать с `@cf/meta/llama-3.2-3b-instruct`.
